In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone

URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
MIN_MAGNITUDE = 3.0
OUTPUT_FILE = "earthquake_full_data_5_years.csv"

# Fetch data month by month to avoid very large API responses.
today = datetime.now(timezone.utc)
start_year = today.year - 5

all_records = []

for year in range(start_year, today.year + 1):
    for month in range(1, 13):
        start_date = datetime(year, month, 1, tzinfo=timezone.utc)

        if start_date >= today:
            break

        if month == 12:
            end_date = datetime(year + 1, 1, 1, tzinfo=timezone.utc)
        else:
            end_date = datetime(year, month + 1, 1, tzinfo=timezone.utc)

        if end_date > today:
            end_date = today

        params = {
            "format": "geojson",
            "starttime": start_date.strftime("%Y-%m-%d"),
            "endtime": end_date.strftime("%Y-%m-%d"),
            "minmagnitude": MIN_MAGNITUDE,
        }

        print(f"Fetching {params['starttime']} to {params['endtime']}...")

        try:
            response = requests.get(URL, params=params, timeout=30)
            response.raise_for_status()
            all_records.extend(response.json()["features"])
        except requests.RequestException as error:
            print(f"Request failed: {error}")

# Convert nested GeoJSON records into a flat DataFrame
df = pd.json_normalize(all_records)

# Rename useful columns clearly
df = df.rename(
    columns={
        "properties.mag": "magnitude",
        "properties.place": "place",
        "properties.time": "time",
        "properties.updated": "updated",
        "properties.tz": "timezone",
        "properties.url": "url",
        "properties.detail": "detail_url",
        "properties.felt": "felt_reports",
        "properties.cdi": "cdi",
        "properties.mmi": "mmi",
        "properties.alert": "alert",
        "properties.status": "status",
        "properties.tsunami": "tsunami",
        "properties.sig": "significance",
        "properties.net": "network",
        "properties.code": "code",
        "properties.ids": "ids",
        "properties.sources": "sources",
        "properties.types": "types",
        "properties.nst": "stations",
        "properties.dmin": "distance_to_station",
        "properties.rms": "rms",
        "properties.gap": "gap",
        "properties.type": "event_type",
        "geometry.coordinates": "coordinates",
        "geometry.type": "geometry_type",
    }
)

# Convert Unix timestamps from milliseconds to readable datetimes
df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)
df["updated"] = pd.to_datetime(df["updated"], unit="ms", utc=True)

# Split [longitude, latitude, depth] into separate columns
df[["longitude", "latitude", "depth_km"]] = pd.DataFrame(
    df["coordinates"].tolist(),
    index=df.index,
)

# Clean comma-delimited identifier columns
for column in ["ids", "sources", "types"]:
    if column in df.columns:
        df[column] = df[column].str.strip(",")

# Drop the now-unneeded nested coordinate column
df = df.drop(columns=["coordinates"])

# Save results
df.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved {len(df):,} earthquake records to {OUTPUT_FILE}")
print(df.head())

Fetching 2021-01-01 to 2021-02-01...
Fetching 2021-02-01 to 2021-03-01...
Fetching 2021-03-01 to 2021-04-01...
Fetching 2021-04-01 to 2021-05-01...
Fetching 2021-05-01 to 2021-06-01...
Fetching 2021-06-01 to 2021-07-01...
Fetching 2021-07-01 to 2021-08-01...
Fetching 2021-08-01 to 2021-09-01...
Fetching 2021-09-01 to 2021-10-01...
Fetching 2021-10-01 to 2021-11-01...
Fetching 2021-11-01 to 2021-12-01...
Fetching 2021-12-01 to 2022-01-01...
Fetching 2022-01-01 to 2022-02-01...
Fetching 2022-02-01 to 2022-03-01...
Fetching 2022-03-01 to 2022-04-01...
Fetching 2022-04-01 to 2022-05-01...
Fetching 2022-05-01 to 2022-06-01...
Fetching 2022-06-01 to 2022-07-01...
Fetching 2022-07-01 to 2022-08-01...
Fetching 2022-08-01 to 2022-09-01...
Fetching 2022-09-01 to 2022-10-01...
Fetching 2022-10-01 to 2022-11-01...
Fetching 2022-11-01 to 2022-12-01...
Fetching 2022-12-01 to 2023-01-01...
Fetching 2023-01-01 to 2023-02-01...
Fetching 2023-02-01 to 2023-03-01...
Fetching 2023-03-01 to 2023-04-01...
F